In [ ]:
import os
import json
import random
import pathlib
from concurrent.futures import ThreadPoolExecutor

In [ ]:
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

## Mount Google Drive in Colab Content

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Set Global ETL Setttings

In [ ]:
SEED = 42
VAL_FRACTION = 0.10
UNKNOWN_PER_CLASS = 30
KAGGLE_DATASET = "kmader/food41"
DOWNLOAD_DIR = pathlib.Path("/content/food41")
OUTPUT_DIR = pathlib.Path("/content/maison_roast_etl")


Food-101 ETL for the Maison-Roast / Basil menu classifier.

Builds a 41-class dataset (40 menu classes + 1 'unknown') from the
kmader/food41 Kaggle dataset, using the official Food-101 train/test split.
Produces train/val/test manifests (relative image paths + integer labels),
a label map, and a verification report.

This are the Classes i am interested to my Vitange E-Commerce Coffe Shop App

In [ ]:
MENU_CLASSES = sorted([
    "apple_pie", "baklava", "beignets", "bread_pudding", "cannoli",
    "carrot_cake", "cheesecake", "chocolate_cake", "chocolate_mousse",
    "churros", "creme_brulee", "cup_cakes", "donuts", "frozen_yogurt",
    "ice_cream", "macarons", "panna_cotta", "red_velvet_cake",
    "strawberry_shortcake", "tiramisu", "waffles",
    "breakfast_burrito", "croque_madame", "eggs_benedict", "french_toast",
    "omelette", "pancakes", "deviled_eggs", "bruschetta",
    "caesar_salad", "cheese_plate", "club_sandwich", "french_onion_soup",
    "garlic_bread", "greek_salad", "grilled_cheese_sandwich",
    "hamburger", "hot_dog", "lasagna", "pizza",
])

 Kaggle authentication from a Colab secret named KAGGLE_TOKEN.
 The secret must hold the full JSON content of kaggle.json, that is
 {"username": "...", "key": "..."}.

In [ ]:

def configure_kaggle():
    from google.colab import userdata
    token = userdata.get("KAGGLE_API_TOKEN")
    kaggle_dir = pathlib.Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    key_path = kaggle_dir / "kaggle.json"
    key_path.write_text(token)
    key_path.chmod(0o600)
    print("Kaggle credentials written to", key_path)


Import Kaggle Secret API KEY TOKEN

In [ ]:
from google.colab import userdata
userdata.get('KAGGLE_API_TOKEN')

Set Kaggle Credentials

In [ ]:
configure_kaggle()

Kaggle credentials written to /root/.kaggle/kaggle.json


##  Download and extract the dataset. Skips if already present.

In [ ]:

def download_dataset():
    if DOWNLOAD_DIR.exists() and any(DOWNLOAD_DIR.rglob("classes.txt")):
        print("Dataset already present, skipping download.")
        return
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    import subprocess
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(DOWNLOAD_DIR), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("Kaggle command failed with stdout:", result.stdout)
        print("Kaggle command failed with stderr:", result.stderr)
        raise RuntimeError("Kaggle download failed. Check token and dataset slug.")
    else:
        print("Kaggle download successful.")


# this is the Kaggle Dataset I am using,  this is  well-know good dataset to teain a Food Classification CNN
https://www.kaggle.com/datasets/kmader/food41

In [ ]:
download_dataset()

Kaggle download successful.


The dataset contains a number of different subsets of the full food-101 data. The idea is to make a more exciting simple training set for image analysis than CIFAR10 or MNIST. For this reason the data includes massively downscaled versions of the images to enable quick tests. The data has been reformatted as HDF5 and specifically Keras HDF5Matrix which allows them to be easily read in. The file names indicate the contents of the file.

In [ ]:
#  Locate images/ and meta/, then load the official metadata.
def locate_dirs():
    if not DOWNLOAD_DIR.exists():
        raise RuntimeError("Dataset not found. Did you download it?")

    classes_txt = next(DOWNLOAD_DIR.rglob("classes.txt"))

    meta_dir = classes_txt.parent

    images_dir = None

    for cand in DOWNLOAD_DIR.rglob("images"):

        if cand.is_dir() and (cand / MENU_CLASSES[0]).exists():
            images_dir = cand
            break

    if images_dir is None:

        raise RuntimeError("Could not locate the images/ directory.")

    return images_dir, meta_dir


In [ ]:
# Map paths a images/{clase}/{id}.jpg, refactor labels:

In [ ]:
def load_metadata(meta_dir):
    meta_dir = pathlib.Path(meta_dir)
    train_json = {}
    test_json = {}
    with open(meta_dir / "train.json") as f:
        train_json = json.load(f)
    with open(meta_dir / "test.json") as f:
        test_json = json.load(f)
    classes = (meta_dir / "classes.txt").read_text().split()

    return train_json, test_json, classes


In [ ]:
IMAGES_DIR, META_DIR = locate_dirs()
TRAIN_JSON, TEST_JSON, ALL_CLASSES = load_metadata(META_DIR)

In [ ]:
print("images dir:", IMAGES_DIR)
print("meta dir:", META_DIR)
print("total classes in metadata:", len(ALL_CLASSES))

images dir: /content/food41/images
meta dir: /content/food41/meta/meta
total classes in metadata: 101


## Verify per-class image counts and sample the resolution spread.

In [ ]:

def verify_counts():

    counts = {cls: len(list((IMAGES_DIR / cls).glob("*.jpg"))) for cls in ALL_CLASSES}

    low = {c: n for c, n in counts.items() if n < 1000}

    print("classes with fewer than 1000 images:", low if low else "none")

    print("min/max images per class:", min(counts.values()), max(counts.values()))

    return counts


Confirm every class has 1000 images as Dataset Card states , critical for train splits and check images resolution as well

In [ ]:

def sample_resolutions(n=500):

    all_rel = [r for ids in TRAIN_JSON.values() for r in ids]

    rng = random.Random(SEED)

    sample = rng.sample(all_rel, min(n, len(all_rel)))

    def dims(rel):
        with Image.open(IMAGES_DIR / (rel + ".jpg")) as im:
            return im.size

    with ThreadPoolExecutor(max_workers=16) as ex:

        sizes = list(ex.map(dims, sample))

    ws = sorted(s[0] for s in sizes)
    hs = sorted(s[1] for s in sizes)

    mid = len(sizes) // 2

    print(f"width  min/median/max: {ws[0]} / {ws[mid]} / {ws[-1]}")
    print(f"height min/median/max: {hs[0]} / {hs[mid]} / {hs[-1]}")
    print("note: originals are variable resolution, resize to 384x384 in training.")


Split logic:
- Menu classes: all their train images go to the train pool, all their test
  images go to the test set.
- Unknown class: UNKNOWN_PER_CLASS random images from each leftover class,
  sampled only from the train split. Never added to the test set.
- The full train pool (menu + unknown) is split VAL_FRACTION into val,
  stratified by label, so unknown also appears in validation.

In [ ]:
COUNTS = verify_counts()
sample_resolutions()

classes with fewer than 1000 images: none
min/max images per class: 1000 1000
width  min/median/max: 288 / 512 / 512
height min/median/max: 277 / 512 / 512
note: originals are variable resolution, resize to 384x384 in training.


## Build the 41-class label map and the leftover class list.

In [ ]:

def build_label_map():

    label_map = {cls: i for i, cls in enumerate(MENU_CLASSES)}
    unknown_label = len(MENU_CLASSES)
    label_map["unknown"] = unknown_label

    return label_map, unknown_label


In [ ]:
LABEL_MAP, UNKNOWN_LABEL = build_label_map()
LEFTOVER_CLASSES = [c for c in ALL_CLASSES if c not in set(MENU_CLASSES)]

In [ ]:
# "leftover classes: 61" && "unknown images: 1830",

In [ ]:
print("menu classes:", len(MENU_CLASSES))
print("leftover classes for unknown:", len(LEFTOVER_CLASSES))
print("unknown images to sample:", len(LEFTOVER_CLASSES) * UNKNOWN_PER_CLASS)


menu classes: 40
leftover classes for unknown: 61
unknown images to sample: 1830


##  Assemble records and make a stratified train/val split.


In [ ]:

def build_train_pool():
    records = []
    for cls in MENU_CLASSES:
        for rel in TRAIN_JSON[cls]:
            records.append((rel + ".jpg", LABEL_MAP[cls], cls))
    rng = random.Random(SEED)
    for cls in LEFTOVER_CLASSES:
        picked = rng.sample(TRAIN_JSON[cls], UNKNOWN_PER_CLASS)
        for rel in picked:
            records.append((rel + ".jpg", UNKNOWN_LABEL, "unknown"))
    return pd.DataFrame(records, columns=["path", "label", "class_name"])


In [ ]:
def build_test_pool():
    records = []
    for cls in MENU_CLASSES:
        for rel in TEST_JSON[cls]:
            records.append((rel + ".jpg", LABEL_MAP[cls], cls))
    return pd.DataFrame(records, columns=["path", "label", "class_name"])



In [ ]:
pool = build_train_pool()
train_df, val_df = train_test_split(
    pool, test_size=VAL_FRACTION, stratify=pool["label"], random_state=SEED
)

In [ ]:
test_df = build_test_pool()
print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))


train: 28647 val: 3183 test: 10000


##  Check referenced files exist, then save manifests and report.

In [ ]:

def count_missing(df):
    return sum(1 for p in df["path"] if not (IMAGES_DIR / p).exists())



In [ ]:

def save_outputs():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
    val_df.to_csv(OUTPUT_DIR / "val.csv", index=False)
    test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)
    index_to_class = {v: k for k, v in LABEL_MAP.items()}
    (OUTPUT_DIR / "label_map.json").write_text(
        json.dumps(
            {"class_to_index": LABEL_MAP, "index_to_class": index_to_class},
            indent=2,
        )
    )
    report = {
        "images_root": str(IMAGES_DIR),
        "num_menu_classes": len(MENU_CLASSES),
        "num_leftover_classes": len(LEFTOVER_CLASSES),
        "unknown_per_class": UNKNOWN_PER_CLASS,
        "unknown_total": len(LEFTOVER_CLASSES) * UNKNOWN_PER_CLASS,
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df),
        "val_fraction": VAL_FRACTION,
        "seed": SEED,
    }
    (OUTPUT_DIR / "etl_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))


In [ ]:
## Better catch it now than during trainign steps

In [ ]:
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{split_name} missing files:", count_missing(split_df))


train missing files: 0
val missing files: 0
test missing files: 0


In [ ]:
save_outputs()

{
  "images_root": "/content/food41/images",
  "num_menu_classes": 40,
  "num_leftover_classes": 61,
  "unknown_per_class": 30,
  "unknown_total": 1830,
  "train_size": 28647,
  "val_size": 3183,
  "test_size": 10000,
  "val_fraction": 0.1,
  "seed": 42
}


## **Resample,  10,000 in test split is too much**
We out to give mroe trianing images , these food images are in difference scales, position so it makes diveristy of single food per classes

Moves TEST_MOVE_PER_CLASS images per menu class out of the official test set
and into the training pool, then re-splits train/val and saves all manifests
to Google Drive.

AIMAGES_DIR, TRAIN_JSON, TEST_JSON, MENU_CLASSES,
LEFTOVER_CLASSES, LABEL_MAP, UNKNOWN_LABEL, UNKNOWN_PER_CLASS, SEED.


Leakage rule: images moved from test are removed from test entirely. An
assertion verifies that train/val and test never share an image.

In [ ]:
import json
import pathlib
import random

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Configuration for this step
TEST_MOVE_PER_CLASS = 150  # 150 * 40 menu classes = 6000 images moved out of test
DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/MaisonRoast/BasilCNN")

In [ ]:
#  Decide which test images move to train and which stay.
def split_test_ids():
    rng = random.Random(SEED + 1)  # independent stream, still reproducible
    moved, kept = {}, {}
    for cls in MENU_CLASSES:
        ids = list(TEST_JSON[cls])
        rng.shuffle(ids)
        moved[cls] = ids[:TEST_MOVE_PER_CLASS]
        kept[cls] = ids[TEST_MOVE_PER_CLASS:]
    return moved, kept

In [ ]:
MOVED_TEST, KEPT_TEST = split_test_ids()
moved_total = sum(len(v) for v in MOVED_TEST.values())

In [ ]:
# Rebuild the pools with the moved images folded into train.
def build_train_pool():

    records = []
    for cls in MENU_CLASSES:

        for rel in TRAIN_JSON[cls]:

            records.append((rel + ".jpg", LABEL_MAP[cls], cls))
        for rel in MOVED_TEST[cls]:
            records.append((rel + ".jpg", LABEL_MAP[cls], cls))

    rng = random.Random(SEED)

    for cls in LEFTOVER_CLASSES:

        for rel in rng.sample(TRAIN_JSON[cls], UNKNOWN_PER_CLASS):
            records.append((rel + ".jpg", UNKNOWN_LABEL, "unknown"))

    return pd.DataFrame(records, columns=["path", "label", "class_name"])


In [ ]:
def build_test_pool():
    records = []
    for cls in MENU_CLASSES:
        for rel in KEPT_TEST[cls]:
            records.append((rel + ".jpg", LABEL_MAP[cls], cls))
    return pd.DataFrame(records, columns=["path", "label", "class_name"])


In [ ]:
pool = build_train_pool()

In [ ]:
train_df, val_df = train_test_split(
    pool, test_size=0.10, stratify=pool["label"], random_state=SEED
)

In [ ]:
test_df = build_test_pool()
print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))


train: 34047 val: 3783 test: 4000


In [ ]:
# Safety check: train/val and test must be disjoint.
def assert_no_leakage(train_df, val_df, test_df):

    train_val = set(train_df["path"]) | set(val_df["path"])
    overlap = train_val & set(test_df["path"])

    assert not overlap, f"LEAKAGE: {len(overlap)} images in both train/val and test"
    print("no leakage: train/val and test are disjoint")


In [ ]:
## Good Catch , it passed,  there isn ot leakage from train set to  test set

In [ ]:
assert_no_leakage(train_df, val_df, test_df)

no leakage: train/val and test are disjoint


In [ ]:
# Save manifests, label map, and report to Drive.

In [ ]:

def save_to_drive():
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(DRIVE_DIR / "train.csv", index=False)
    val_df.to_csv(DRIVE_DIR / "val.csv", index=False)
    test_df.to_csv(DRIVE_DIR / "test.csv", index=False)
    index_to_class = {v: k for k, v in LABEL_MAP.items()}
    (DRIVE_DIR / "label_map.json").write_text(
        json.dumps(
            {"class_to_index": LABEL_MAP, "index_to_class": index_to_class},
            indent=2,
        )
    )
    report = {
        "images_root": str(IMAGES_DIR),
        "num_menu_classes": len(MENU_CLASSES),
        "num_leftover_classes": len(LEFTOVER_CLASSES),
        "unknown_per_class": UNKNOWN_PER_CLASS,
        "unknown_total": len(LEFTOVER_CLASSES) * UNKNOWN_PER_CLASS,
        "test_moved_per_class": TEST_MOVE_PER_CLASS,
        "test_moved_total": moved_total,
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df),
        "val_fraction": 0.10,
        "seed": SEED,
    }
    (DRIVE_DIR / "etl_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))


In [ ]:
## Confirm it is safe on my Drive

In [ ]:
save_to_drive()


{
  "images_root": "/content/food41/images",
  "num_menu_classes": 40,
  "num_leftover_classes": 61,
  "unknown_per_class": 30,
  "unknown_total": 1830,
  "test_moved_per_class": 150,
  "test_moved_total": 6000,
  "train_size": 34047,
  "val_size": 3783,
  "test_size": 4000,
  "val_fraction": 0.1,
  "seed": 42
}


In [ ]:
!ls -l /content/drive/MyDrive/MaisonRoast/BasilCNN

total 1567
-rw------- 1 root root     313 Aug 28 21:57 etl_report.json
-rw------- 1 root root    1996 Aug 28 21:57 label_map.json
-rw------- 1 root root  153902 Aug 28 21:57 test.csv
-rw------- 1 root root 1302705 Aug 28 21:57 train.csv
-rw------- 1 root root  144666 Aug 28 21:57 val.csv
